# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

We start by examining the distributions of our core signals and the target label. Most search metrics are heavily right-tailed; a small number of pages capture the majority of the volume. We establish a "development floor" of 500 impressions to ensure we are auditing signals on pages with enough data to be statistically meaningful.

In this professional audit, we evaluate how these signals correlate with **future decline** (the target label).

In [12]:
import os
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

def load_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
    return os.environ.get('HF_TOKEN')

HF_TOKEN = load_token()

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN environment variable is not set. Please set it before running the script."
    )



# Setup DuckDB for Warehouse Access
def get_warehouse_data(t0_date):
    conn = duckdb.connect(database=":memory:")
    conn.execute("INSTALL httpfs; LOAD httpfs;")
    conn.execute("CREATE SECRET (TYPE HUGGINGFACE, TOKEN ?)", [HF_TOKEN])
    print("Successfully authenticated with Hugging Face and loaded warehouse data.")
    sql = f"""
    WITH 
    feature_window AS (
        SELECT 
            content_hash_id,
            SUM(gsc_impressions) as impressions_90d,
            SUM(gsc_clicks) as clicks_90d,
            SUM(ga4_sessions) as sessions_90d,
            SUM(sessions_ai) as ai_sessions_90d,
            SUM(ga4_engaged_sessions) as engaged_sessions_90d,
            SUM(scroll_events) as scroll_events_90d,
            AVG(gsc_avg_position) as avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) as days_with_impressions,
            COUNT(DISTINCT CASE WHEN ga4_sessions > 0 THEN report_date END) as days_with_sessions
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        WHERE report_date BETWEEN '{t0_date}'::DATE - INTERVAL 90 DAYS AND '{t0_date}'::DATE
        GROUP BY content_hash_id
    ),
    target_window AS (
        SELECT 
            content_hash_id,
            SUM(gsc_impressions) as impressions_future
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        WHERE report_date BETWEEN '{t0_date}'::DATE + INTERVAL 1 DAY AND '{t0_date}'::DATE + INTERVAL 30 DAYS
        GROUP BY content_hash_id
    ),
    content_dims AS (
        SELECT 
            content_hash_id as content_id, 
            client_hash_id as client_id, 
            search_volume, competition, cpc, word_count, char_count, content_type, main_intent
        FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    SELECT 
        d.*,
        f.*,
        CASE 
            WHEN t.impressions_future < (f.impressions_90d / 3.0) THEN 1 
            ELSE 0 
        END as is_declining_label
    FROM content_dims d
    JOIN feature_window f ON d.content_id = f.content_hash_id
    JOIN target_window t ON d.content_id = t.content_hash_id;
    """
    return conn.execute(sql).df()

# Use Training T0 as default for the audit
T0_AUDIT = "2026-04-30"
frame = get_warehouse_data(T0_AUDIT)
print(f"Successfully loaded warehouse data for T0={T0_AUDIT}")


Successfully authenticated with Hugging Face and loaded warehouse data.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully loaded warehouse data for T0=2026-04-30


## 2. Signal test #1 / #2 / #3 (verdict each)

We test three core search signals against the **future decline label** to determine if they can serve as reliable heuristics for our baseline score:
1. **Impression Volume**: We check if pages with higher volume are more or less likely to decline (Verdict: MIXED).
2. **Position Tier**: We verify if deeper pages are more prone to further decline than top-tier pages (Verdict: MIXED).
3. **Content Type**: We look for stable differences in decline rates between article types (Verdict: MIXED).

In [16]:
# 1. Data Cleaning & Tiers
# Calculate position tier for the audit
def get_position_tier(pos):
    if pos == 0: return 'no_data'
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

frame['position_tier'] = frame['avg_position'].apply(get_position_tier)

print(f'Aggregated content-client rows: {len(frame):,}')
print(f'Unique content items: {frame.content_id.nunique():,}')
print(f'Rows at or above the 500-impression development floor: {(frame.impressions_90d >= 500).sum():,}')
print(frame[['impressions_90d', 'clicks_90d', 'sessions_90d', 'is_declining_label', 'avg_position']].describe(percentiles=[.5, .9, .99]).round(3))

Aggregated content-client rows: 362,179
Unique content items: 362,179
Rows at or above the 500-impression development floor: 91,246
       impressions_90d  clicks_90d  sessions_90d  is_declining_label  \
count       362179.000  362179.000    306402.000          362179.000   
mean          2100.300       6.281         8.948               0.370   
std          10199.678      50.679        86.162               0.483   
min              0.000       0.000         0.000               0.000   
50%             10.000       0.000         0.000               0.000   
90%           4023.000       8.000        13.000               1.000   
99%          36984.220     117.000       169.000               1.000   
max        1519291.000   14344.000     38217.000               1.000   

       avg_position  
count    221623.000  
mean         15.975  
std          16.217  
min           0.000  
50%           9.393  
90%          37.529  
99%          75.716  
max         308.000  


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption regarding future decline?*

In [17]:
eligible = frame[(frame.impressions_90d >= 500) & (frame.position_tier != 'no_data')].copy()
eligible['impression_bin'] = pd.qcut(eligible['impressions_90d'], q=4, duplicates='drop')

# Test 1: Volume vs Decline
volume_test = eligible.groupby('impression_bin', observed=True).agg(
    n=('content_id', 'size'), 
    decline_rate=('is_declining_label', 'mean')
)
print('Signal 1 — volume quartiles and decline rate:')
print(volume_test.round(3))
print('Verdict: MIXED - volume serves as a data-strength guardrail rather than a causal driver of decline.')

# Test 2: Position vs Decline
position_test = eligible.groupby('position_tier', observed=True).agg(
    n=('content_id', 'size'), 
    decline_rate=('is_declining_label', 'mean')
)
print('\nSignal 2 — position tiers and decline rate:')
print(position_test.round(3))
ordered = position_test.reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep']).dropna()
print('Verdict: MIXED - position indicates current visibility, but decline risk exists across all tiers.')

# Test 3: Content Type vs Decline
type_test = eligible.groupby(['position_tier', 'content_type'], dropna=False, observed=True).agg(
    n=('content_id', 'size'), 
    decline_rate=('is_declining_label', 'mean')
)
print('\nSignal 3 — content type within position cells with n >= 50:')
print(type_test[type_test['n'] >= 50].round(3).head(30))
print('Verdict: MIXED unless stable differences persist in adequately sized cells.')

Signal 1 — volume quartiles and decline rate:
                         n  decline_rate
impression_bin                          
(499.999, 1112.0]    22815         0.643
(1112.0, 2686.0]     22809         0.687
(2686.0, 7764.0]     22811         0.718
(7764.0, 1519291.0]  22811         0.713
Verdict: MIXED - volume serves as a data-strength guardrail rather than a causal driver of decline.

Signal 2 — position tiers and decline rate:
                   n  decline_rate
position_tier                     
deep            1296         0.474
page_1         45916         0.661
page_3_5       17456         0.700
striking       22032         0.738
top_3           4546         0.781
Verdict: MIXED - position indicates current visibility, but decline risk exists across all tiers.

Signal 3 — content type within position cells with n >= 50:
                                      n  decline_rate
position_tier content_type                           
deep          keyword article      1290         0.4

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [18]:
# Flag-linked test — high-risk candidates within a valid position tier
eligible['below_position_median'] = eligible['avg_position'] > eligible.groupby('position_tier', observed=True)['avg_position'].transform('median')
flag_test = eligible.groupby('below_position_median', observed=True).agg(
    n=('content_id', 'size'), 
    decline_rate=('is_declining_label', 'mean')
)
print('Flag-linked test — pages with below-median position within a valid position tier:')
print(flag_test.round(3))

assert eligible['impressions_90d'].ge(500).all()
assert eligible['position_tier'].ne('no_data').all()
print('The measured rule supports a review queue for sufficiently observed, visible pages; it does not establish that an edit will cause recovery.')

Flag-linked test — pages with below-median position within a valid position tier:
                           n  decline_rate
below_position_median                     
False                  45623         0.692
True                   45623         0.688
The measured rule supports a review queue for sufficiently observed, visible pages; it does not establish that an edit will cause recovery.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.